In [0]:
%pip install xgboost

In [ ]:
import json
import mlflow
import xgboost as xgb
import pandas as pd
from typing import Iterator, Tuple
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import DoubleType, StructType, StructField, IntegerType

mlflow.set_registry_uri("databricks-uc")

# --- Load model from Unity Catalog using the champion alias ---
MODEL_NAME = "stocks.models.xgb_direction_predictor"
client = mlflow.MlflowClient()
model_version = client.get_model_version_by_alias(MODEL_NAME, "champion")
run = client.get_run(model_version.run_id)
feature_cols = json.loads(run.data.params["feature_cols"])
model_uri = f"models:/{MODEL_NAME}@champion"

print(f"Model version : {model_version.version}")
print(f"Feature count : {len(feature_cols)}")

wrapper_model = mlflow.xgboost.load_model(model_uri)
native_booster = wrapper_model.get_booster()
model_bytes = native_booster.save_raw()

# --- Data ---
gold_df = spark.table("stocks.gold.stocks_w_prev_returns").filter(
    col("Date") >= "2026-01-17"
)

schema = StructType([
    StructField("prediction", IntegerType(), True),
    StructField("certainty",  DoubleType(),  True),
])

@pandas_udf(schema)
def predict_bytes_udf(iterator: Iterator[Tuple[pd.Series, ...]]) -> Iterator[pd.DataFrame]:
    booster = xgb.Booster()
    booster.load_model(bytearray(model_bytes))
    for args in iterator:
        X = pd.concat(args, axis=1)
        X.columns = feature_cols
        probs = booster.predict(xgb.DMatrix(X))
        yield pd.DataFrame({"prediction": (probs > 0.5).astype(int), "certainty": probs})

gold_df_scored = gold_df.withColumn(
    "model_output", predict_bytes_udf(*[col(c) for c in feature_cols])
)
final_df = gold_df_scored.select(
    "Date", "company",
    "model_output.prediction", "model_output.certainty",
    "label",
)

In [0]:

final_df.display()

In [0]:

from delta.tables import DeltaTable

if not spark.catalog.tableExists(f"stocks.reporting.predictions"):
    final_df.write.mode("overwrite").format("delta").saveAsTable(f"stocks.reporting.predictions"
        )

delta_table = DeltaTable.forName(spark, f"stocks.reporting.predictions")

delta_table.alias("t").merge(
    final_df.alias("s"),
    "s.Date = t.Date AND s.company = t.company"
).whenNotMatchedInsertAll().execute()


# 2. Load the NEW Source (The reference data)
source_df = spark.table("stocks.gold.stocks_w_prev_returns")

# 3. Execute the Merge
delta_table.alias("p") \
  .merge(
    source_df.alias("s"),
    condition = "p.Date = s.Date AND p.company = s.company"
  ) \
  .whenMatchedUpdate(
    # Update 'label' in predictions using the value from 'gold'
    set = { "label": col("s.label") }
  ) \
  .execute()